*  DSC 540-T302 Data Preparation
*  Week 11 Exercise
*  Peter Lozano

# Activity 8.01 Retrieving Data Accurately from Databases

For this activity, I will be fetching data and retrieve information from two tables, `persons` and `pets`, which are part of the `petsdb` database.

## Import libraries

In [1]:
import sqlite3
import pandas as pd

## Connect to the database and verify connection

I will connect to the database using the `sqlite3` library and verify the connection by doing a simple query. I placed the `petsdb` database in a folder named `Week 11 Data` in my OneDrive.

In [2]:
# Connect to the database
conn = sqlite3.connect('Week 11 Data/petsdb')

# Simple query to test
query_persons = "SELECT * FROM persons"

# Execute the query and handle potential errors
try:
    conn.execute(query_persons)
except sqlite3.Error as e:
    print(f"An error occurred: {e}")
else:
    print("Connection to the database is successful.")

# Close the connection
conn.close()

Connection to the database is successful.


Now that I have successfully connected to the database, I will proceed to answer the questions using SQL queries.

## Question 1: What is the count of people belonging to different age groups in the `persons` table?

I will need to create a for loop for this since I will need to count the number of people for each age in the database. 

In [3]:
# Reconnect to the database to fetch ages
conn = sqlite3.connect('Week 11 Data/petsdb')
query = "SELECT count(*) as count, age FROM persons GROUP BY age"
try:
    for people, age in conn.execute(query):
        print(f'We have {people} people who are {age} years old.')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

We have 2 people who are 5 years old.
We have 1 people who are 6 years old.
We have 1 people who are 7 years old.
We have 3 people who are 8 years old.
We have 1 people who are 9 years old.
We have 2 people who are 11 years old.
We have 3 people who are 12 years old.
We have 1 people who are 13 years old.
We have 4 people who are 14 years old.
We have 2 people who are 16 years old.
We have 2 people who are 17 years old.
We have 3 people who are 18 years old.
We have 1 people who are 19 years old.
We have 3 people who are 22 years old.
We have 2 people who are 23 years old.
We have 3 people who are 24 years old.
We have 2 people who are 25 years old.
We have 1 people who are 27 years old.
We have 1 people who are 30 years old.
We have 3 people who are 31 years old.
We have 1 people who are 32 years old.
We have 1 people who are 33 years old.
We have 2 people who are 34 years old.
We have 3 people who are 35 years old.
We have 3 people who are 36 years old.
We have 1 people who are 37 ye

## Question 2: Which age group has the maximum number of people?

I can use the same for loop but order the results by count and limit the results to 1 to get the age group with the maximum number of people. I will just have to adjust the print statement

In [4]:
query = "SELECT count(*) as count, age FROM persons GROUP BY age ORDER BY count DESC LIMIT 1"
try:
    for people, age in conn.execute(query):
        print(f'The highest number of people is {people} and came from {age} age group.')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

The highest number of people is 5 and came from 73 age group.


## Question 3: How many people do not have a last name?

I can still use the same structure for the for loop but, the query will need to be adjusted to include the field for last name. Also, the for loop will need to only index the count field since I will not be grouping by age for this question.

In [5]:
query = "SELECT count(*) as count FROM persons WHERE last_name IS NULL"
try:
    for people in conn.execute(query):
        # Print only the count of people without a last name
        print(f'{people[0]}')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

60


Slightly different from the book but, I think it is better to have an integer than a dictionary for this question since I am only interested in the count of people without a last name.

## Question 4: How many people have more than one pet?

This will require a bit of a different approach on the query since I will need to utilize the `owner_id` field to group by and count the distinct pets for each owner. I will also need to use a `HAVING` clause to filter the results to only include those with more than one pet. This will need to be wrapped in as a subquery to get the count of people with more than one pet. If not, I will return a list of counts of pets for each owner instead of the total count of people with more than one pet.

In [6]:
# Main query to select the count of people with more than one pet
query = "SELECT count(*) FROM \
    (SELECT count(owner_id) FROM pets GROUP BY owner_id HAVING count(owner_id) > 1)" # Subquery to count pets per owner and filter those with more than one pet
try:
    for people in conn.execute(query):
        print(f'{people[0]} people have more than one pet.')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

43 people have more than one pet.


## Question 5: How many pets have received treatment?

I have a field within the database named `treatment_done` that indicates whether a pet has received treatment or not. I will use this field to count the number of pets that have received treatment by filtering for records where `treatment_done` is marked as **1** for yes.

In [7]:
# Main query counts pets that have received treatment
query = "SELECT count(*) FROM pets WHERE treatment_done = 1"
try:
    # Changing loop variable to 'row' to utilize a more generic term
    for row in conn.execute(query):
        print(f'{row[0]} ')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

36 


## Question 6: How many pets have received treatment, and the type of pet is known?

The book indicates that the `pet_type` field contains null values for some records. Since there is limited information on exactly what this field pertains to in terms of pet type, I will assume that the null values are the `unknown` types.

In [8]:
# Main query counts pets that have received treatment and pet type is known
query = "SELECT count(*) FROM pets WHERE treatment_done = 1 AND pet_type IS NOT NULL"
try:
    for row in conn.execute(query):
        print(f'{row[0]} ')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

16 


## Question 7: How many pets are from the city called east port?

So the way that these data tables are structured, there is no direct link between the `pets` table and the city information. However, the `persons` table contains the city information for each person, and the `pets` table contains the `owner_id` field that links to the `persons` table using the `id` field. Therefore, I will need to perform a join between the two tables to get the desired information.

In [9]:
# Main query counts pets that are from the city called east port
query = "SELECT count(*) FROM pets \
    JOIN persons ON pets.owner_id = persons.id \
    WHERE persons.city = 'east port'" # Join pets and persons tables to filter by city
try:
    for row in conn.execute(query):
        print(f'{row[0]} ')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

49 


## Question 8: How many pets are from the city called east port, and who received treatment?

This query will be similar to the previous 2 exercises. I will be joining the `pets` and `persons` tables on the `owner_id` and `id` fields, respectively. However, I will also need to add a filter to the query to only include pets that have received treatment by checking the `treatment_done` field for a value of **1** for yes.

In [10]:
# Main query counts pets that are from the city called east port
query = "SELECT count(*) FROM pets \
    JOIN persons ON pets.owner_id = persons.id \
    WHERE persons.city = 'east port' \
    AND pets.treatment_done = 1" # Join pets and persons tables to filter by city and treatment
try:
    for row in conn.execute(query):
        print(f'{row[0]} ')
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

# Close the connection
conn.close()

11 


Lastly, I close the connection to the database since this concludes the activity.